| 方法                         | 用途             | 示例                                |
| -------------------------- | -------------- | --------------------------------- |
| `OmegaConf.create()`       | 创建配置           | `OmegaConf.create({"a": 1})`      |
| `OmegaConf.load()`         | 从文件加载          | `OmegaConf.load("config.yaml")`   |
| `OmegaConf.save()`         | 保存到文件          | `OmegaConf.save(cfg, "out.yaml")` |
| `OmegaConf.merge()`        | 合并多个配置         | `OmegaConf.merge(cfg1, cfg2)`     |
| `OmegaConf.resolve()`      | 解析插值           | `OmegaConf.resolve(cfg)`          |
| `OmegaConf.to_container()` | 转 Python dict  | `OmegaConf.to_container(cfg)`     |
| `OmegaConf.to_yaml()`      | 转 YAML 字符串     | `OmegaConf.to_yaml(cfg)`          |
| `OmegaConf.select()`       | 安全选择路径         | `OmegaConf.select(cfg, "a.b")`    |
| `OmegaConf.update()`       | 更新值            | `OmegaConf.update(cfg, "a", 2)`   |
| `OmegaConf.set()`          | 设置值            | `OmegaConf.set(cfg, "a.b", 3)`    |
| `OmegaConf.is_dict()`      | 检查是否为 dict     | `OmegaConf.is_dict(cfg)`          |
| `OmegaConf.is_list()`      | 检查是否为 list     | `OmegaConf.is_list(cfg)`          |
| `OmegaConf.set_struct()`   | 启用结构保护         | `OmegaConf.set_struct(cfg, True)` |
| `OmegaConf.structured()`   | 从 dataclass 创建 | `OmegaConf.structured(MyConfig)`  |


#  OmegaConf
OmegaConf 是 Hydra 的底层配置库，提供强大的 YAML/字典配置操作能力。

## 基础实例：创建和访问配置

### 创建配置对象

In [ ]:
from omegaconf import OmegaConf, DictConfig, ListConfig

# 方式 1：从字典创建
dict_cfg = OmegaConf.create({
    "model": {
        "name": "resnet50",
        "num_classes": 1000,
        "layers": [64, 128, 256, 512]
    },
    "training": {
        "lr": 0.001,
        "epochs": 10,
        "batch_size": 32
    }
})

# 方式 2：从 YAML 字符串创建
yaml_cfg = OmegaConf.create("""
model:
  name: resnet50
  num_classes: 1000
training:
  lr: 0.001
  epochs: 10
""")

# 方式 3：从文件加载
file_cfg = OmegaConf.load("config.yaml")

# 方式 4：合并多个配置
merged = OmegaConf.merge(dict_cfg, yaml_cfg)
print(merged)

## 核心方法详解与实例
###  访问和修改配置

In [2]:
cfg = OmegaConf.create({
    "model": {"name": "resnet50", "layers": [64, 128, 256]},
    "training": {"lr": 0.001, "optimizer": "adam"}
})

# ─── 访问 ───
# 属性访问（推荐）
print(cfg.model.name)           # "resnet50"

# 字典访问（兼容 dict）
print(cfg["model"]["name"])     # "resnet50"

# 嵌套安全访问
print(cfg.training.get("momentum", 0.9))  # 0.9（默认值）

# ─── 修改 ───
cfg.training.lr = 0.01          # 属性修改
cfg["training"]["epochs"] = 20  # 字典式修改

# 添加新键
cfg.training.weight_decay = 1e-4
# 结果: training: {lr: 0.01, optimizer: adam, weight_decay: 0.0001}

# 结构化标志：禁止添加新键（可选）
OmegaConf.set_struct(cfg, True)
cfg.new_key = "value"  # 报错！ConfigKeyError

resnet50
resnet50
0.9


ConfigAttributeError: Key 'new_key' is not in struct
    full_key: new_key
    object_type=dict

### 变量插值（Interpolation）
最强大特性：配置内引用其他值

In [ ]:
cfg = OmegaConf.create({
    "paths": {
        "data_dir": "/data/imagenet",
        "checkpoint_dir": "/checkpoints"
    },
    "model": {
        "save_path": "${paths.checkpoint_dir}/resnet50.pth",  # 引用其他键
        "data_path": "${paths.data_dir}"                       # 直接引用
    },
    "training": {
        "log_dir": "${paths.checkpoint_dir}/logs/${model.name}",  # 嵌套引用
        "lr": 0.001,
        "lr_decay": "${training.lr}"  # 自引用（复制值）
    }
})

# 解析前
print(cfg.model.save_path) # 未解析其中变量"${paths.checkpoint_dir}/resnet50.pth"（字符串）

# 解析插值
# 注意：resolve 是原地修改，且不可逆
OmegaConf.resolve(cfg)
print(cfg.model.save_path) # "/checkpoints/resnet50.pth"（实际值）
print(cfg.training.log_dir) # "/checkpoints/logs/resnet50"

#### 动态插值实例：



In [ ]:
# 运行时动态计算
cfg = OmegaConf.create({
    "batch": {"size": 32, "num_workers": 4},
    "total_samples": "${batch.size} * ${batch.num_workers} * 10"  # 表达式
})

OmegaConf.register_new_resolver('eval',lamda x : eval(str(x)))

### 合并配置（Merge）

In [3]:
# 基础配置
base_cfg = OmegaConf.create({
    "model": {"name": "resnet50", "pretrained": True},
    "training": {"lr": 0.001, "epochs": 10}
})

# 实验特定覆盖
exp_cfg = OmegaConf.create({
    "model": {"name": "resnet101"},  # 覆盖 name，保留 pretrained
    "training": {"epochs": 20, "batch_size": 64}  # 覆盖 epochs，新增 batch_size
})

# 合并（后覆盖前）
merged = OmegaConf.merge(base_cfg, exp_cfg)
print(OmegaConf.to_yaml(merged))

model:
  name: resnet101
  pretrained: true
training:
  lr: 0.001
  epochs: 20
  batch_size: 64



#### 多配置合并策略（Hydra 实际使用）：

In [ ]:
# 模拟 Hydra 的 defaults 机制
defaults = [
    OmegaConf.load("conf/model/resnet.yaml"),
    OmegaConf.load("conf/optimizer/adam.yaml"),
    OmegaConf.load("conf/data/cifar10.yaml"),
]
final_cfg = OmegaConf.merge(*defaults)

### 转换为标准 Python 对象


In [4]:
cfg = OmegaConf.create({
    "model": {"name": "transformer", "layers": 12},
    "gpus": [0, 1, 2, 3],
    "distributed": True
})

# ─── to_container ───
# 转换为 dict/list（递归）
container = OmegaConf.to_container(cfg)
print(type(container))        # <class 'dict'>
print(type(container["model"]))  # <class 'dict'>

# 保留 OmegaConf 对象（不转换）
raw = OmegaConf.to_container(cfg, resolve=False)

# ─── to_yaml ───
yaml_str = OmegaConf.to_yaml(cfg)
print(yaml_str)
# 输出：
# model:
#   name: transformer
#   layers: 12
# gpus:
# - 0
# - 1
# - 2
# - 3
# distributed: true

# ─── to_json ───
import json
json_str = json.dumps(OmegaConf.to_container(cfg))

<class 'dict'>
<class 'dict'>
model:
  name: transformer
  layers: 12
gpus:
- 0
- 1
- 2
- 3
distributed: true



### 选择性访问和更新

In [6]:
cfg = OmegaConf.create({
    "model": {
        "encoder": {"type": "transformer", "hidden_size": 768},
        "decoder": {"type": "gpt", "layers": 12}
    },
    "training": {
        "stage1": {"lr": 0.001, "epochs": 5},
        "stage2": {"lr": 0.0001, "epochs": 10}
    }
})

# ─── select ───
# 选择子树
encoder_cfg = OmegaConf.select(cfg, "model.encoder")
print(encoder_cfg.type)  # "transformer"

# 带默认值
scheduler = OmegaConf.select(cfg, "training.scheduler", default="cosine")
print(scheduler)  # "cosine"（不存在时返回默认值）

# ─── update ───
# 批量更新
OmegaConf.update(cfg, "training.stage1.lr", 0.002, merge=False)  # 直接替换
OmegaConf.update(cfg, "training.stage3", {"lr": 0.00001}, merge=True)  # 合并



print(cfg)

transformer
cosine
{'model': {'encoder': {'type': 'transformer', 'hidden_size': 768}, 'decoder': {'type': 'gpt', 'layers': 12}}, 'training': {'stage1': {'lr': 0.002, 'epochs': 5}, 'stage2': {'lr': 0.0001, 'epochs': 10}, 'stage3': {'lr': 1e-05}}}


### 列表操作

In [7]:
cfg = OmegaConf.create({
    "transforms": [
        {"name": "Resize", "size": 256},
        {"name": "CenterCrop", "size": 224},
        {"name": "ToTensor"}
    ],
    "gpus": [0, 1, 2, 3]
})

# 访问列表
print(cfg.transforms[0].name)  # "Resize"

# 列表追加
cfg.transforms.append({"name": "Normalize", "mean": [0.485, 0.456, 0.406]})

# 列表合并
new_transforms = [
    {"name": "RandomFlip"},
    {"name": "ColorJitter"}
]
cfg.transforms = OmegaConf.merge(cfg.transforms, new_transforms)

# 检查类型
print(isinstance(cfg.transforms, ListConfig))  # True
print(OmegaConf.is_list(cfg.transforms))       # True
print(OmegaConf.is_dict(cfg.model))            # True

Resize
True
True


ConfigAttributeError: Missing key model
    full_key: model
    object_type=dict

###  环境变量和 CLI 解析集成

In [ ]:
import os

# ─── 从环境变量创建 ───
os.environ["MODEL_NAME"] = "bert-base"
os.environ["BATCH_SIZE"] = "32"

cfg = OmegaConf.create({
    "model": {"name": "${env:MODEL_NAME, default_model}"},  # 带默认值
    "training": {"batch_size": "${env:BATCH_SIZE, 16}"}
})

OmegaConf.resolve(cfg)
print(cfg.model.name)        # "bert-base"
print(cfg.training.batch_size)  # "32"（注意：是字符串！）

# ─── 类型转换 ───
from omegaconf import SI  # String interpolation

cfg = OmegaConf.create({
    "batch_size": SI("${oc.decode:${env:BATCH_SIZE, 16}}")  # 解码为 int
})
OmegaConf.resolve(cfg)
print(type(cfg.batch_size))  # <class 'int'>

### 差异比较（Diff）

In [ ]:
cfg1 = OmegaConf.create({"a": 1, "b": {"c": 2, "d": 3}})
cfg2 = OmegaConf.create({"a": 1, "b": {"c": 99, "e": 4}})

# 计算差异
diff = OmegaConf.to_container(OmegaConf.merge(cfg1, cfg2))
# 或使用第三方库 deepdiff

# OmegaConf 内置：masked_copy（选择特定键）
masked = OmegaConf.masked_copy(cfg1, ["a", "b.c"])
print(OmegaConf.to_yaml(masked))
# 输出：
# a: 1
# b:
#   c: 2

### 实际应用：ML 实验配置管理

In [8]:
from omegaconf import OmegaConf
from dataclasses import dataclass

@dataclass
class ModelConfig:
    name: str = "resnet50"
    num_classes: int = 1000
    pretrained: bool = True

@dataclass
class TrainConfig:
    model: ModelConfig = ModelConfig()
    lr: float = 0.001
    epochs: int = 10

# 结构化配置（类型安全）
cfg = OmegaConf.structured(TrainConfig)

# 从 CLI 字符串覆盖（模拟 Hydra 行为）
overrides = ["model.name=resnet101", "lr=0.01"]
for override in overrides:
    key, value = override.split("=")
    OmegaConf.update(cfg, key, value, merge=False)

# 类型检查（可选）
OmegaConf.set_struct(cfg, True)

# 保存实验配置
OmegaConf.save(cfg, "experiment.yaml")

# 打印最终配置
print(OmegaConf.to_yaml(cfg))

model:
  name: resnet101
  num_classes: 1000
  pretrained: true
lr: 0.01
epochs: 10

